## Step 1. Load dataset from the Databricks metastore

In this step, we load the `mental_health_lifestyle_dataset` table
directly from the default Databricks database.
Since the table schema is already defined, we can immediately
inspect its structure and confirm that the data is available for analysis.



In [0]:
# Load the table from the default database
df = spark.table("default.mental_health_lifestyle_dataset")

# Print schema to confirm data types
df.printSchema()

# Count total records
record_count = df.count()
print(f"Total records: {record_count}")

# Show sample data
df.show(5, truncate=False)



root
 |-- Country: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Exercise Level: string (nullable = true)
 |-- Diet Type: string (nullable = true)
 |-- Sleep Hours: double (nullable = true)
 |-- Stress Level: string (nullable = true)
 |-- Mental Health Condition: string (nullable = true)
 |-- Work Hours per Week: integer (nullable = true)
 |-- Screen Time per Day (Hours): double (nullable = true)
 |-- Social Interaction Score: double (nullable = true)
 |-- Happiness Score: double (nullable = true)

Total records: 3000
+---------+---+------+--------------+----------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|Country  |Age|Gender|Exercise Level|Diet Type |Sleep Hours|Stress Level|Mental Health Condition|Work Hours per Week|Screen Time per Day (Hours)|Social Interaction Score|Happiness Score|
+---------+---+------+--------------+-------

## Step 2. Data quality check – Missing values

Before starting any analysis, it is essential to check the data quality.  
Here we look for missing (NULL) values in each column of the dataset.  
This helps identify where cleaning or imputation might be needed.


In [0]:
from pyspark.sql.functions import col, sum

# Calculate number of missing values per column
missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)


+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|Country|Age|Gender|Exercise Level|Diet Type|Sleep Hours|Stress Level|Mental Health Condition|Work Hours per Week|Screen Time per Day (Hours)|Social Interaction Score|Happiness Score|
+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|0      |0  |0     |0             |0        |0          |0           |0                      |0                  |0                          |0                       |0              |
+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+



### Interpretation

- Columns with **0 missing values** are fully complete.  
- Columns with **non-zero values** may require attention before further analysis.  
- In this case we have 0 missing values for all the fields


## Step 3. Descriptive statistics

In this step, we explore basic descriptive statistics for the main numerical variables.  
We use Spark SQL to calculate averages and standard deviations for age, sleep, work hours, screen time,  
social interaction, and happiness.

This helps us understand the general lifestyle trends in the dataset.


In [0]:
spark.sql("""
SELECT
    ROUND(AVG(Age), 1) AS avg_age,
    ROUND(STDDEV(Age), 1) AS std_age,
    ROUND(AVG(`Sleep Hours`), 2) AS avg_sleep,
    ROUND(STDDEV(`Sleep Hours`), 2) AS std_sleep,
    ROUND(AVG(`Work Hours per Week`), 1) AS avg_work,
    ROUND(AVG(`Screen Time per Day (Hours)`), 2) AS avg_screen,
    ROUND(AVG(`Social Interaction Score`), 2) AS avg_social,
    ROUND(AVG(`Happiness Score`), 2) AS avg_happiness,
    ROUND(STDDEV(`Happiness Score`), 2) AS std_happiness
FROM default.mental_health_lifestyle_dataset
""").show()


+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+
|avg_age|std_age|avg_sleep|std_sleep|avg_work|avg_screen|avg_social|avg_happiness|std_happiness|
+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+
|   41.2|   13.4|     6.48|      1.5|    39.5|      5.09|      5.47|          5.4|         2.56|
+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+



## Step 3.1. Descriptive breakdown by gender

Next, we check how these lifestyle indicators differ between genders.  
This helps us understand if there are any notable patterns in habits or well-being.


In [0]:
# Display average happiness by gender
display(
    spark.sql("""
    SELECT Gender, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY Gender
    ORDER BY avg_happiness DESC
    """)
)



Gender,avg_happiness
Male,5.47
Other,5.43
Female,5.29


Databricks visualization. Run in Databricks to view.

### Step 3.2. Descriptive breakdown by country

Here we look at the top countries by average happiness score.  
This gives a quick overview of regional lifestyle differences.


In [0]:
display(
    spark.sql("""
    SELECT Country, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY Country
    HAVING COUNT(*) > 20
    ORDER BY avg_happiness DESC
    LIMIT 10
    """)
)



Country,avg_happiness
Canada,5.56
Australia,5.49
India,5.38
Germany,5.37
USA,5.35
Brazil,5.34
Japan,5.28


Databricks visualization. Run in Databricks to view.

## Step 4. Lifestyle factors and well-being

In this step, we explore how lifestyle variables such as exercise level, diet type,
and sleep hours relate to happiness and mental health condition.

We use Spark SQL to compute averages and distributions, 
and display results as interactive charts suitable for the dashboard.


### Average Happiness by Exercise Level

This bar chart shows how happiness varies with exercise frequency.
Higher activity levels may correspond to higher well-being.


In [0]:
display(
    spark.sql("""
    SELECT `Exercise Level`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Exercise Level`
    ORDER BY avg_happiness DESC
    """)
)


Exercise Level,avg_happiness
High,5.55
Moderate,5.36
Low,5.29


Databricks visualization. Run in Databricks to view.

### Average Happiness by Diet Type

This bar chart explores whether dietary choices are associated with higher happiness.
It can reveal patterns between different diet types and well-being.


In [0]:
display(
    spark.sql("""
    SELECT `Diet Type`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Diet Type`
    ORDER BY avg_happiness DESC
    """)
)


Diet Type,avg_happiness
Vegetarian,5.66
Junk Food,5.44
Keto,5.34
Vegan,5.29
Balanced,5.25


Databricks visualization. Run in Databricks to view.

### Sleep Hours Buckets vs Average Happiness

To better understand how different amounts of sleep affect well-being,
we group participants into 1-hour sleep buckets.
This allows us to see clear trends between sleep duration and happiness.



In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Sleep Hours`)
    ORDER BY sleep_bucket
    """)
)



sleep_bucket,avg_happiness,count_participants
1,6.17,3
2,5.78,30
3,5.31,116
4,5.25,338
5,5.26,595
6,5.44,792
7,5.45,626
8,5.59,347
9,5.16,125
10,5.65,24


Databricks visualization. Run in Databricks to view.

### Stress Level vs Average Happiness

This bar chart shows the inverse relationship between stress levels and happiness.
Higher stress may correspond to lower well-being.



In [0]:
display(
    spark.sql("""
    SELECT `Stress Level`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Stress Level`
    ORDER BY avg_happiness ASC
    """)
)



Stress Level,avg_happiness
Moderate,5.34
Low,5.41
High,5.44


Databricks visualization. Run in Databricks to view.

### Work Hours Buckets vs Average Happiness

We group weekly work hours into 1-hour buckets to understand their effect on happiness.
This approach simplifies the visualization and highlights trends more clearly.


In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Work Hours per Week`) AS work_hours_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Work Hours per Week`)
    ORDER BY work_hours_bucket
    """)
)


work_hours_bucket,avg_happiness,count_participants
20,5.43,64
21,5.77,76
22,5.43,70
23,5.85,83
24,5.74,66
25,4.49,57
26,5.44,94
27,5.4,88
28,5.37,68
29,5.07,61


Databricks visualization. Run in Databricks to view.

### Screen Time Buckets vs Average Happiness

We group daily screen time into 1-hour buckets to explore its relationship with happiness.
Bucketing helps to compare participants with different screen habits in a clear way.


In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Screen Time per Day (Hours)`) AS screen_hours_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Screen Time per Day (Hours)`)
    ORDER BY screen_hours_bucket
    """)
)


screen_hours_bucket,avg_happiness,count_participants
2,5.47,449
3,5.27,481
4,5.34,470
5,5.33,501
6,5.4,531
7,5.54,537
8,5.42,31


Databricks visualization. Run in Databricks to view.

## Step 5. Combined Lifestyle Factors vs Happiness

In this step, we explore how multiple lifestyle factors interact to influence happiness.  
We group participants by:
- Exercise Level
- Diet Type
- Sleep Hours (buckets of 1 hour)

We calculate the average happiness for each combination to identify patterns and trends.


In [0]:
display(
    spark.sql("""
    SELECT
        `Exercise Level`,
        --`Diet Type`,
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Exercise Level`, FLOOR(`Sleep Hours`)
    HAVING COUNT(*) >= 5
    ORDER BY `Exercise Level`, sleep_bucket
    """)
)


Exercise Level,sleep_bucket,avg_happiness,count_participants
High,2,6.4,5
High,3,4.72,38
High,4,5.11,102
High,5,5.57,182
High,6,5.67,269
High,7,5.54,221
High,8,5.83,104
High,9,6.0,40
High,10,3.33,6
Low,2,5.29,12


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.sql("""
    SELECT
        --`Exercise Level`,
        `Diet Type`,
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Diet Type`, FLOOR(`Sleep Hours`)
    HAVING COUNT(*) >= 5
    ORDER BY `Diet Type`, sleep_bucket
    """)
)

Diet Type,sleep_bucket,avg_happiness,count_participants
Balanced,2,5.07,9
Balanced,3,5.33,27
Balanced,4,5.16,69
Balanced,5,4.92,136
Balanced,6,5.43,158
Balanced,7,5.64,136
Balanced,8,4.87,65
Balanced,9,4.53,20
Junk Food,2,6.59,10
Junk Food,3,6.31,21


Databricks visualization. Run in Databricks to view.

## Step 6. Heatmap: Sleep Buckets vs Happiness by Exercise Level

We create a pivot table to visualize average happiness across sleep buckets and exercise levels.  
This heatmap allows us to quickly spot trends and interactions between sleep and activity on well-being.


In [0]:
# Pivot table: rows = Sleep Bucket, columns = Exercise Level, values = avg_happiness
pivot_df = spark.sql("""
SELECT 
    FLOOR(`Sleep Hours`) AS sleep_bucket,
    `Exercise Level`,
    ROUND(AVG(`Happiness Score`),2) AS avg_happiness
FROM default.mental_health_lifestyle_dataset
GROUP BY FLOOR(`Sleep Hours`), `Exercise Level`
""").groupBy("sleep_bucket").pivot("Exercise Level").avg("avg_happiness")

# Display as interactive heatmap/table
display(pivot_df.orderBy("sleep_bucket"))


sleep_bucket,High,Low,Moderate
1,null,1.5,8.5
2,6.4,5.29,5.98
3,4.72,5.38,5.87
4,5.11,5.63,5.03
5,5.57,5.18,5.07
6,5.67,5.15,5.51
7,5.54,5.38,5.42
8,5.83,5.46,5.52
9,6.0,4.5,5.05
10,3.33,6.38,6.46
